# Error analysis

~200 stratified errors (high-confidence FP, high-confidence FN, near-threshold), read manually and clustered into named failure modes with counts and examples.

## Sample stratified errors

In [ ]:
import sys
sys.path.insert(0, "../src")

import pandas as pd

from error_analysis import flag_likely_label_noise, stratified_error_sample

# Point this at whichever run's val predictions you want to analyze --
# written by src/baseline.py (baseline_tfidf_val_predictions.csv) or
# src/train.py (<out_name>_val_predictions.csv), both under reports/.
PREDICTIONS_CSV = "../reports/deberta_multitask_val_predictions.csv"

val_df = pd.read_csv(PREDICTIONS_CSV)
print(f"loaded {len(val_df):,} val rows")

sample = stratified_error_sample(val_df, n_per_stratum=70, seed=42)
print(sample["error_stratum"].value_counts())

noise_flag = flag_likely_label_noise(val_df)
print(f"\nlikely-label-noise flagged (heuristic, read manually to confirm): "
      f"{noise_flag.sum():,} / {len(val_df):,} ({noise_flag.mean():.1%})")

sample.to_csv("../reports/error_sample_for_review.csv", index=False)
print("\nWrote ../reports/error_sample_for_review.csv -- open it (or the cells below) and read every row.")


This part is inherently manual: read `../reports/error_sample_for_review.csv` (or loop the cell below) row by row and assign each one a failure-mode label. Starting taxonomy from the project spec -- add/split/merge categories as the real errors dictate, don't force-fit:

- `reclaimed_slur_or_ingroup` -- reclaimed slurs / in-group speech flagged as an attack
- `quoted_toxicity` -- someone reporting/quoting abuse gets flagged for the abuse they're reporting
- `sarcasm_irony` -- sarcasm or irony misread as sincere toxicity
- `counter_speech` -- arguing against racism/bigotry, flagged as the thing it's arguing against
- `label_noise` -- model looks right on a re-read; this is annotator disagreement, not a model error (cross-check against `flag_likely_label_noise` above)
- `politics_as_toxicity` -- heated but legitimate political disagreement scored high
- `other` -- doesn't fit the above; note what it actually looks like

Tag rows in the cell below (`row_index -> label`), then run the aggregation cell to produce the counts table for the README.

In [ ]:
# Print each row for manual reading. Re-run with an offset/slice to work
# through the sample in batches.
for idx, row in sample.iloc[0:20].iterrows():
    print(f"[{idx}] stratum={row['error_stratum']} target={row['target']:.2f} pred={row['prediction']:.3f}")
    print(f"    {row['comment_text'][:300]}")
    print()


In [ ]:
# Fill in as you read: {row_index: failure_mode_label}. This dict is the
# actual analysis -- everything after it is just aggregation.
failure_mode_labels: dict[int, str] = {
    # 147: "sarcasm_irony",
    # 191: "quoted_toxicity",
}

sample["failure_mode"] = sample.index.map(failure_mode_labels)
labeled = sample.dropna(subset=["failure_mode"])
print(f"labeled {len(labeled)} / {len(sample)} sampled rows so far")


## Failure-mode table (-> README)

In [ ]:
if labeled.empty:
    print("No rows tagged yet -- fill in failure_mode_labels above (read the "
          "printed rows, or ../reports/error_sample_for_review.csv) and re-run.")
    failure_mode_table = pd.DataFrame(columns=["count", "example_1", "example_2"])
else:
    failure_mode_table = (
        labeled.groupby("failure_mode")
        .apply(lambda g: pd.Series({
            "count": len(g),
            "example_1": g["comment_text"].iloc[0][:200],
            "example_2": g["comment_text"].iloc[1][:200] if len(g) > 1 else "",
        }), include_groups=False)
        .sort_values("count", ascending=False)
    )
    print(failure_mode_table)
    failure_mode_table.to_csv("../reports/failure_mode_table.csv")

# Cross-check: what fraction of ALL sampled errors (not just the ones
# manually tagged so far) does the label_noise heuristic flag? If a
# meaningful share of "errors" are actually annotator disagreement, the
# model's real ceiling is lower than the raw metric suggests -- say so in
# the README rather than letting the headline number imply otherwise.
noise_in_sample = flag_likely_label_noise(sample)
print(f"\nheuristic label-noise rate within the sampled errors: {noise_in_sample.mean():.1%} "
      f"({noise_in_sample.sum()} / {len(sample)})")
